# 1. Deep ensembles, calibration, and uncertainty

Train three independent classifiers, combine their predictions with `Ensemble`,
calibrate the combined distribution, and inspect uncertainty.

**Run:** install the editable package and Jupyter as described in [the examples guide](README.md),
then restart the kernel and run all cells. This notebook uses only CPU and synthetic data.

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from torch import nn

from bensemble.calibration.scaling import TemperatureScaling
from bensemble.core.ensemble import Ensemble
from bensemble.metrics import (
    expected_calibration_error,
    negative_log_likelihood,
    reliability_diagram,
)
from bensemble.uncertainty import decompose_classification_uncertainty

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)  # Small CPU workloads avoid thread-pool overhead.
DEVICE = torch.device("cpu")
NUM_MEMBERS = 3
EPOCHS = 250

## 1. Separate training, calibration, and testing

The validation split is reserved for fitting the temperature. Test labels are used
only for the final evaluation. Standardization uses training statistics alone.

In [ ]:
# Generate two noisy crescents using only PyTorch.
generator = torch.Generator().manual_seed(SEED)
n = 900
labels = torch.arange(n) % 2
angle = torch.rand(n, generator=generator) * torch.pi
features = torch.stack((angle.cos(), angle.sin()), dim=1)
features[labels == 1, 0] = 1 - features[labels == 1, 0]
features[labels == 1, 1] = 0.5 - features[labels == 1, 1]
features += 0.25 * torch.randn(n, 2, generator=generator)

indices = torch.randperm(n, generator=generator)
split_indices = {
    "train": indices[:540],
    "validation": indices[540:720],
    "test": indices[720:],
}
x_train, y_train = features[split_indices["train"]], labels[split_indices["train"]]
x_val, y_val = (
    features[split_indices["validation"]],
    labels[split_indices["validation"]],
)
x_test, y_test = features[split_indices["test"]], labels[split_indices["test"]]

# Fit preprocessing on training data only, then reuse it on the held-out sets.
train_mean = x_train.mean(dim=0)
train_std = x_train.std(dim=0).clamp_min(1e-6)
x_train = (x_train - train_mean) / train_std
x_val = (x_val - train_mean) / train_std
x_test = (x_test - train_mean) / train_std

plt.figure(figsize=(6, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap="coolwarm", s=12)
plt.xlabel("Standardized x1")
plt.ylabel("Standardized x2")
plt.title("Training data")
plt.show()

## 2. Train independent models

Each model receives a fresh initialization and its own optimizer. The training
loop is ordinary PyTorch; `Ensemble` does not train its members.

In [ ]:
models = []
for member in range(NUM_MEMBERS):
    torch.manual_seed(SEED + member)
    model = nn.Sequential(
        nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2)
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    model.train()
    for _ in range(EPOCHS):
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(x_train), y_train)
        loss.backward()
        optimizer.step()
    models.append(model.eval())

ensemble = Ensemble.from_models(models).eval()
print(f"Trained {ensemble.num_members} models")

## 3. Average probabilities, then calibrate

`predict_members` returns `(members, batch, classes)`. The default `ensemble(x)`
averages raw model outputs, which are logits here. For a predictive mixture we
instead apply softmax **to each member**, then average probabilities.

We fit temperature scaling to the **log of the mixture probabilities** on validation
data. These are valid logits for the mixture. Calibration is therefore applied
after aggregation, consistently on validation and test data.

In [ ]:
with torch.no_grad():
    val_probs = ensemble.predict_members(x_val).softmax(dim=-1).mean(dim=0)
    member_probs = ensemble.predict_members(x_test).softmax(dim=-1)
    test_probs = member_probs.mean(dim=0)

scaler = TemperatureScaling(init_temp=1.0)
scaler.fit(val_probs.clamp_min(1e-8).log(), y_val)
with torch.no_grad():
    calibrated_probs = scaler(test_probs.clamp_min(1e-8).log()).softmax(dim=-1)

assert member_probs.shape == (NUM_MEMBERS, len(x_test), 2)
assert torch.isfinite(calibrated_probs).all()
print(f"Fitted temperature: {scaler.temperature.item():.3f}")
for name, probs in [("Original", test_probs), ("Calibrated", calibrated_probs)]:
    accuracy = (probs.argmax(dim=-1) == y_test).float().mean().item()
    print(
        f"{name}: accuracy={accuracy:.3f}, "
        f"NLL={negative_log_likelihood(probs, y_test):.3f}, "
        f"ECE={expected_calibration_error(probs, y_test):.3f}"
    )

## 4. Inspect calibration and disagreement

Calibration minimizes validation NLL; it does not guarantee lower ECE or NLL on
every test sample. Empty reliability bins are omitted from the plot.

Uncertainty below is computed from the **original member probabilities**, before
mixture-level calibration: predictive entropy = mean member entropy + mutual
information. These are entropy quantities, not regression variances. Disagreement
alone is not a guarantee of out-of-distribution detection.

In [ ]:
total, aleatoric, epistemic = decompose_classification_uncertainty(member_probs)
assert torch.allclose(total, aleatoric + epistemic, atol=1e-6)
print(f"Mean predictive entropy: {total.mean():.3f}")
print(f"Mean member entropy: {aleatoric.mean():.3f}")
print(f"Mean epistemic uncertainty: {epistemic.mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, probs, title in zip(
    axes, [test_probs, calibrated_probs], ["Original", "Calibrated"]
):
    diagram = reliability_diagram(probs, y_test, n_bins=10)
    occupied = [
        i for i, proportion in enumerate(diagram["proportions"]) if proportion > 0
    ]
    ax.plot([0, 1], [0, 1], "k--")
    ax.plot(
        [diagram["confidences"][i] for i in occupied],
        [diagram["accuracies"][i] for i in occupied],
        "o-",
    )
    ax.set(
        xlim=(0, 1), ylim=(0, 1), xlabel="Confidence", ylabel="Accuracy", title=title
    )
plt.tight_layout()
plt.show()

Next: [MC Dropout](02_mc_dropout.ipynb) builds an ensemble from one stochastic model.